<div class="align-center">

<a href="https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/index.html"><img src="https://raw.githubusercontent.com/ROCm/gpuaidev/main/docs/images/rocm_logo.png" alt="ROCm AI Developer Hub" width="150" style="display:inline-block; margin-right: 20px;"></a>
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115" style="display:inline-block;"></a>

</div>

<div align="center">

<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

</div>

---

# Pokémon LLM Agent with Unsloth on AMD ROCm (companion notebook)

**Authors:** Yue Yuan ([yueyuan](https://github.com/yueyuan), yueyuan@amd.com), Bill He ([billishyahao](https://github.com/billishyahao), bill.he@amd.com)

**Phase 13 — Pokémon LLM Agent with Unsloth.** Build a Pokemon Showdown action model from raw replay logs using `Qwen/Qwen3-4B`, Unsloth, and AMD ROCm. For the **AMD AI Developer Hub**-style walkthrough with extended prerequisites and eval protocol text, use `pokemon_llm_agent_unsloth_rocm_tutorial.ipynb` in this repo.

### What you will build
1. **Portable ROCm setup**: Configure a notebook that works on MI300X-class systems and still degrades gracefully on smaller AMD machines.
2. **Disk-safe data preparation**: Stream and filter raw replay logs from Hugging Face without materializing the full 40GB+ corpus locally.
3. **Quick validation SFT**: Run a short 50-step alignment loop and watch the model shift from generic chat behavior to valid `move` / `switch` actions.
4. **Inference sanity check**: Test the tuned model on a real battle prefix and inspect whether it emits a legal next action.
5. **Export and publish**: Package the merged checkpoint and push your tutorial artifact to Hugging Face.

### Tutorial scope
This notebook intentionally trains on a small streamed subset so the workflow stays teachable, reproducible, and safe on limited disk. A **longer-trained reference** can be published after training or loaded from the Hugging Face model card linked in the release tutorial.

### Prerequisites
To install Unsloth on your AMD machine, follow the [AMD ROCm installation guide](https://unsloth.ai/docs/get-started/install/amd). This notebook is adapted from the Unsloth notebook ecosystem and keeps the same [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) notebook license context.

## 1. AMD ROCm environment setup
When running on AMD GPUs such as MI300X or Radeon PRO W7900, Unsloth relies on ROCm plus PyTorch SDPA kernels for strong throughput. The next cell sets conservative defaults before importing training libraries, prefers large external cache mounts when they exist, and falls back to `/tmp` so the tutorial does not break on machines without `/shared-docker`.

In [10]:
import os
from pathlib import Path

import datasets

cache_candidates = [
    "/shared-docker/.cache/huggingface",
    "/data/huggingface",
    "/tmp/pokemon-hf-cache",
]
tmp_candidates = [
    "/shared-docker/.cache/tmp",
    "/data/tmp",
    "/tmp/pokemon-tmp",
]

cache_root = next((path for path in cache_candidates if os.path.isdir(path)), cache_candidates[-1])
tmp_root = next((path for path in tmp_candidates if os.path.isdir(path)), tmp_candidates[-1])

# Default to the first visible AMD GPU unless the user already pinned a device.
os.environ.setdefault("HIP_VISIBLE_DEVICES", "0")
os.environ.setdefault("ROCR_VISIBLE_DEVICES", os.environ["HIP_VISIBLE_DEVICES"])

os.environ["HF_HOME"] = cache_root
os.environ["HF_DATASETS_CACHE"] = f"{cache_root}/datasets"
os.environ["TMPDIR"] = tmp_root

Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)

datasets.config.HF_DATASETS_CACHE = os.environ["HF_DATASETS_CACHE"]

# RDNA3 cards often need the AOTriton flag for Flash Attention via SDPA.
os.environ.setdefault("TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL", "1")
os.environ.setdefault("PYTORCH_HIP_ALLOC_CONF", "expandable_segments:False")
os.environ.setdefault("UNSLOTH_SKIP_TORCHVISION_CHECK", "1")

print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE={os.environ['HF_DATASETS_CACHE']}")
print(f"TMPDIR={os.environ['TMPDIR']}")


## 2. Installation
If you are not already inside a prepared ROCm image, install Unsloth and the core training stack. For maximum AMD compatibility, this tutorial avoids depending on optional 8-bit optimizer packages and sticks to the standard PyTorch / Transformers / TRL toolchain.

In [11]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install --no-cache-dir --no-deps unsloth unsloth_zoo
    !pip install --no-cache-dir transformers accelerate peft trl datasets psutil sentencepiece protobuf tyro huggingface_hub hf_transfer einops
else:
    pass  # Colab / Kaggle usually need their own setup flow.

## 3. Load the base model
We load `Qwen/Qwen3-4B` with a configuration tuned for AMD inference stability. A full-scale training recipe uses longer contexts and more data; this tutorial keeps the setup compact enough for a short reproducible run.

In [12]:
import unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Small enough for a fast tutorial, large enough for real replay prefixes.
dtype = torch.bfloat16  # Recommended on AMD for stable training and inference.
load_in_4bit = False    # Keep inference stable on ROCm for the public tutorial flow.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    attn_implementation="sdpa",
)

==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    AMD Instinct MI300X VF. Num GPUs = 1. Max memory: 191.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.1. ROCm Toolkit: 7.1.25424. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.56s/it]


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [13]:
# 2. Apply LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

## 4. Data preparation from raw replay logs
The **raw-log** idea is simple but demanding: feed the model messy real replay logs instead of hand-written state summaries. For the public tutorial we stream `milkkarten/pokemon-showdown-replays-merged`, filter to stronger Gen 9 games, and build a compact subset entirely in memory so the notebook stays disk-safe.

In [ ]:
from datasets import Dataset, load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

MIN_RATING = 1400
MAX_TRAIN_SAMPLES = 2000
SHUFFLE_BUFFER = 10_000
MAX_LOG_CHARS = 6000

SYSTEM_TEMPLATE = (
    "You are a Pokemon Showdown battle AI. You play as {side}. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)

def extract_winner_side(log_text):
    winner = None
    players = {}
    for line in log_text.split("\n"):
        parts = line.split("|")
        if len(parts) >= 4 and parts[1] == "player":
            players[parts[2]] = parts[3]
        if len(parts) >= 3 and parts[1] == "win":
            winner = parts[2]
    if not winner:
        return None
    for side, name in players.items():
        if name == winner:
            return side
    return None

def format_sample(example):
    log_text = example["log"]
    side = extract_winner_side(log_text)
    if not side:
        return {"text": ""}

    lines = log_text.strip().split("\n")
    turn_positions = []
    for i, line in enumerate(lines):
        parts = line.split("|")
        if len(parts) >= 3 and parts[1] == "turn":
            try:
                turn_positions.append((int(parts[2]), i))
            except ValueError:
                pass

    if len(turn_positions) < 2:
        return {"text": ""}

    # Keep prompts compact for the tutorial by using the first actionable turn
    # after the opening whenever possible.
    target_turn_idx = 0 if len(turn_positions) == 2 else 1
    _, turn_line_idx = turn_positions[target_turn_idx]
    next_turn_line = turn_positions[target_turn_idx + 1][1] if target_turn_idx + 1 < len(turn_positions) else len(lines)

    action = None
    for j in range(turn_line_idx + 1, next_turn_line):
        parts = lines[j].split("|")
        if len(parts) < 4:
            continue
        if parts[1] == "move" and parts[2].startswith(f"{side}a:"):
            tera = ""
            start_look = max(0, j - 3)
            end_look = min(len(lines), j + 3)
            if any("terastallize" in lines[k] and side in lines[k] for k in range(start_look, end_look)):
                tera = " terastallize"
            action = f"move {parts[3]}{tera}"
            break
        if parts[1] == "switch" and parts[2].startswith(f"{side}a:"):
            pokemon = parts[2].split(": ")[1] if ": " in parts[2] else parts[2]
            action = f"switch {pokemon}"
            break

    if not action:
        return {"text": ""}

    log_prefix = "\n".join(lines[:turn_line_idx + 1])
    if len(log_prefix) > MAX_LOG_CHARS:
        return {"text": ""}

    messages = [
        {"role": "system", "content": SYSTEM_TEMPLATE.format(side=side)},
        {"role": "user", "content": log_prefix},
        {"role": "assistant", "content": action},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

print("Streaming, filtering, and formatting replay logs...")
dataset = load_dataset("milkkarten/pokemon-showdown-replays-merged", split="train", streaming=True)
dataset = dataset.shuffle(seed=3407, buffer_size=SHUFFLE_BUFFER)
dataset = dataset.filter(lambda x: "gen9" in str(x.get("formatid") or "") and (x.get("rating") or 0) >= MIN_RATING)

train_samples = []
scanned = 0
for row in dataset:
    scanned += 1
    formatted = format_sample(row)
    if formatted["text"]:
        train_samples.append(formatted)
    if len(train_samples) >= MAX_TRAIN_SAMPLES:
        break

train_dataset = Dataset.from_list(train_samples).shuffle(seed=3407)
print(f"Collected {len(train_dataset)} training examples after scanning {scanned} streamed replays.")
print(train_dataset[0]["text"][:800])

Repo card metadata block was not found. Setting CardData to empty.
[huggingface_hub.repocard|WARNING]Repo card metadata block was not found. Setting CardData to empty.


Generating train split:  96%|█████████▌| 27837802/29057184 [08:06<00:21, 57181.98 examples/s]


DatasetGenerationError: An error occurred while generating the dataset

## 5. Quick validation SFT
Now we run a deliberately short 50-step supervised fine-tuning loop. The point is not to maximize ladder strength in one notebook session, but to make *Agentic Alignment* visible: even a tiny run is often enough to push the model toward strict action formatting instead of free-form commentary.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=50,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="model_output_v6_demo",
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)

trainer_stats = trainer.train()

# Optional: export a standalone merged checkpoint for sharing or Hub upload.
# export_dir = "model_output_v6_demo/merged_model"
# model.save_pretrained_merged(export_dir, tokenizer, save_method="merged_16bit")

## 6. Inference sanity check
Use the same prompt once before training and once after training if you want to see the alignment jump clearly.

This cell prints the **same legality checks** as `pokemon_llm_agent_unsloth_rocm_tutorial.ipynb`, via `scripts/eval/showdown_agent_eval.py`:

- **structure_ok / single_line_ok:** compact `move …` / `switch …` line.
- **request_present / move_legal_in_request:** if the log includes a Showdown `|request|{…}` line, we verify the move name against **non-disabled** `active[].moves` (tutorial demo log includes a synthetic request; `Earthquake` is **not** in the list on purpose so bad guesses show `move_legal_in_request=False`).
- **switch_legal_in_request:** bench targets from `side.pokemon` when a request exists.

Reload the notebook file in Jupyter (**File → Reload from disk**) if you still see only `strict_action_format`.


In [ ]:
import sys
from pathlib import Path

import torch

_repo = Path.cwd().resolve()
for base in [_repo, *_repo.parents]:
    if (base / "scripts" / "eval" / "showdown_agent_eval.py").is_file():
        sys.path.insert(0, str(base / "scripts" / "eval"))
        break
else:
    raise FileNotFoundError(
        "Could not find scripts/eval/showdown_agent_eval.py — set Jupyter cwd to the pokemon repo root."
    )

from showdown_agent_eval import (
    postprocess_agent_response,
    tutorial_demo_log_with_request,
    validate_action_against_log,
)

model.eval()
FastLanguageModel.for_inference(model)

sys_msg = (
    "You are a Pokemon Showdown battle AI. You play as p2. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)
user_msg = tutorial_demo_log_with_request()

messages = [
    {"role": "system", "content": sys_msg},
    {"role": "user", "content": user_msg},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
if "<|im_start|>assistant\n" in full_response:
    response = full_response.split("<|im_start|>assistant\n")[-1]
else:
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
response = postprocess_agent_response(response)

first_line = response.splitlines()[0].strip() if response.strip() else ""
checks = validate_action_against_log(first_line, user_msg)

print("\n--- AI AGENT PREDICTION ---")
print(response)
print("--- legality & format (showdown_agent_eval) ---")
print(f"structure_ok={checks['structure_ok']}")
print(f"single_line_ok={checks['single_line_ok']}")
print(f"request_present={checks['request_present']}")
if checks["move_name_nonempty"] is not None:
    print(f"move_name_nonempty={checks['move_name_nonempty']}")
if checks["switch_target_ok"] is not None:
    print(f"switch_target_ok={checks['switch_target_ok']}")
if checks["move_legal_in_request"] is not None:
    print(f"move_legal_in_request={checks['move_legal_in_request']}")
if checks["switch_legal_in_request"] is not None:
    print(f"switch_legal_in_request={checks['switch_legal_in_request']}")
if checks.get("parsed") and checks["parsed"].get("tera") and checks["tera_legal_in_request"] is not None:
    print(f"tera_legal_in_request={checks['tera_legal_in_request']}")
print(f"notes={checks['notes']}")


## 7. Export and publish to Hugging Face
After the 50-step tutorial run, you can optionally merge the adapter into standalone weights and publish the result.

```python
export_dir = "model_output_v6_demo/merged_model"
model.save_pretrained_merged(export_dir, tokenizer, save_method="merged_16bit")
```

Then upload the merged folder and notebook:

```bash
hf auth whoami
hf upload-large-folder your-username/pokemon-showdown-agent-tutorial-demo model_output_v6_demo/merged_model
hf upload your-username/pokemon-showdown-agent-tutorial-demo pokemon_agent_demo_notebook_v2.ipynb pokemon_llm_agent_unsloth_rocm_tutorial.ipynb
```

A published reference checkpoint is on Hugging Face (`GoldenGrapeGentleman1/pokemon-showdown-agent-v6`); this notebook remains a lighter flow for fast reproduction.

## Bonus: advanced tactics with GRPO

Supervised fine-tuning teaches the model how strong players act. To push beyond imitation and reward outcomes or formatting more explicitly, you can add **Group Relative Policy Optimization (GRPO)** on top.

For Pokemon, a first reward function can stay simple: reward valid action formatting, reward strategically sensible outputs, and penalize clearly bad or impossible commands. The cell below is still conceptual, but it shows how the agent can extend into RL after the SFT tutorial.

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import re

# 1. Define a Reward Function (Example: Check if output contains a valid command)
def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        # TRL completions may be strings or list of dicts depending on version
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        
        # Reward the model heavily if it successfully follows the format `move X` or `switch X`
        cmd_match = re.search(r"(move\s+[\w\-]+|switch\s+[\w\-]+)", text, re.IGNORECASE)
        if cmd_match:
            rewards.append(5.0)
        else:
            rewards.append(-3.0) # Penalize conversational babbling
            
    return rewards

# 2. Prepare GRPO Prompts (We only need the prompts, GRPO will generate its own completions)
grpo_prompts = [{"prompt": p["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"} for p in train_samples[:50]]
from datasets import Dataset
grpo_dataset = Dataset.from_list(grpo_prompts)

# 3. Configure GRPO
grpo_config = GRPOConfig(
    output_dir = "grpo_outputs",
    learning_rate = 3e-6,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4, # Generate 4 different strategies to compare
    max_completion_length = 128,
    temperature = 1.3,   # Encourage exploration
    max_steps = 10,
    logging_steps = 1,
    report_to = "none",
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
)

grpo_trainer = GRPOTrainer(
    model = model,
    reward_funcs = [format_reward_func],
    args = grpo_config,
    train_dataset = grpo_dataset,
)

# Uncomment to run the GRPO loop
# grpo_trainer.train()